In [ ]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# General

In [ ]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from mdbmaster import MasterParams
from sys import prefix
mp = MasterParams(verbose=True)
io = FileIO()

# DB-Specific

In [ ]:
from mdblib import albumoftheyear
mio   = albumoftheyear.MusicDBIO(verbose=True)
db    = mio.db

# Perm Dir

In [ ]:
def setupPermDir(db):
    mp = MasterParams(verbose=False)
    prefixDir = DirInfo(prefix)
    projDir   = prefixDir.join(mp.getProjectName())
    projDir.mkDir()
    libDir    = projDir.join("mdblib")
    libDir.mkDir()
    permDBDir = libDir.join(db)
    permDBDir.mkDir()
    return permDBDir
permDBDir = setupPermDir(db)
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

In [ ]:
from webUtils import webUtil, getHTML
web = webUtil()

In [ ]:
class listItem:
    def __init__(self, div):
        urlref    = div.find("a", {"itemprop": "url"})
        self.url  = urlref.attrs['href'] if urlref is not None else None
        self.name = urlref.text if urlref is not None else None
        
        position = div.find("span", {"itemprop": "position"})
        self.position = position.text if position is not None else None
        
        release  = div.find("div", {"class": "albumListDate"})
        self.release = release.text if release is not None else None
        
        genre    = div.find("div", {"class": "albumListGenre"})
        genreRef = genre.findAll('a') if genre is not None else []
        self.genre = [{ref.text: ref.attrs['href']} for ref in genreRef]
        
        score = div.find("div", {"class": "albumListScoreContainer"})
        score = score.find('div', {"class": "scoreValue"}) if score is not None else None
        self.score = score.text if score is not None else None
        
        links    = div.find("div", {"class": "albumListLinks"})
        self.links = {ref.attrs['data-track-action']: ref.attrs['href'] for ref in links.findAll('a')}

    def get(self):
        return self.__dict__
    

In [ ]:
from time import sleep
for year in range(2021,1977,-1):
    retval = web.download("https://www.albumoftheyear.org/lists.php?y={0}".format(year))
    io.save(idata=retval.getData(), ifile="/Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/AlbumOfTheYear/aoty/{0}.p".format(year))
    sleep(3)

In [ ]:
from fsUtils import fileUtil,mkDir
from time import sleep
from fileIO import fileIO
io = fileIO()

for year in range(2020,1977,-1):
    print("="*50,year,"="*50)
    bsdata = getHTML('/Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/AlbumOfTheYear/aoty/{0}.p'.format(year))
    
    
    refs = bsdata.findAll("a")
    listRefs  = [ref for ref in refs if '/list/' in ref.attrs.get('href', '') and ref.find('img') is not None]
    listNames = {ref.attrs['href']: ref.find('img').attrs['alt'] for ref in listRefs}
    print("  ==> Found {0} lists".format(len(listNames)))
    
    
    toget = {}
    mkDir('/Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/AlbumOfTheYear/aoty/{0}'.format(year))
    for href,name in listNames.items():
        url = "https://www.albumoftheyear.org{0}".format(href)
        savename = "/Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/AlbumOfTheYear/aoty/{0}/{1}.p".format(year,name.replace("/", "-"))
        if not fileUtil(savename).exists:
            toget[url] = savename
    print("There are {0} lists to download".format(len(toget)))

    
    for i,(url,savename) in enumerate(toget.items()):
        if fileUtil(savename).exists:
            continue
        print("{0: <10}{1}".format("{0}/{1}".format(i,len(toget)), url))
        retval = web.download(url)
        if retval.getCode() == 200:
            io.save(idata=retval.getData(), ifile=savename)
            sleep(3)
        else:
            print("Error with {0}".format(url))
            sleep(10)

In [ ]:
io.save(idata=retval.getData(), ifile="aotylists.p")

In [ ]:
bsdata = getHTML("aotylists.p") #retval.getData())

In [ ]:
from fsUtils import fileUtil
toget = {}
for href,name in listNames.items():
    url = "https://www.albumoftheyear.org{0}".format(href)
    savename = "/Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/AlbumOfTheYear/aoty/{0}.p".format(name.replace("/", "-"))
    if not fileUtil(savename).exists:
        toget[url] = savename
print("There are {0} lists to download".format(len(toget)))

In [ ]:
from time import sleep
from fileIO import fileIO
io = fileIO()
for i,(url,savename) in enumerate(toget.items()):
    if fileUtil(savename).exists:
        continue
    print("{0: <10}{1}".format("{0}/{1}".format(i,len(toget)), url))
    retval = web.download(url)
    if retval.getCode() == 200:
        io.save(idata=retval.getData(), ifile=savename)
        sleep(3)
    else:
        print("Error with {0}".format(url))
        sleep(10)

In [ ]:
retval.getCode()

In [ ]:
from glob import glob
from fsUtils import fileUtil

listData = {}
for i,ifile in enumerate(glob("/Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/AlbumOfTheYear/aoty/*/*")):
    fu = fileUtil(ifile)
    year, name = fu.parents[0], fu.basename
    if listData.get(year) is None:
        listData[year] = {}    

    print("{0: <6}{1: <75}".format(year,name), end="")
    bsdata        = getHTML(ifile)
    albumsListDiv = bsdata.find("div", {"itemtype": "http://schema.org/ItemList"})
    albumsList    = albumsListDiv.findAll("div", {"class": "albumListRow"})    
    albumsData    = [listItem(div).get() for div in albumsList]
    
    print(len(albumsData))
    listData[year][name] = albumsData

In [ ]:
from fileIO import fileIO
io = fileIO()
io.save(idata=listData, ifile="/Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/AlbumOfTheYear/aoty/listData.p")

# 

In [ ]:
from ioutils import FileIO
io = FileIO()
listData = io.get("/Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/AlbumOfTheYear/aoty/listData.p")

In [ ]:
listData.keys()

In [ ]:
#['1978']["Paste's 30 Best Albums of 1978"]
retval = {}
for year,yearData in listData.items():
    for chart,chartData in yearData.items():
        for record in chartData:
            name  = record['name']
            links = record['links']
            if isinstance(links,dict):
                key = links.get('Spotify')
                if key is None:
                    continue
                retval[key.split("/")[-1]] = name

In [ ]:
from pandas import Series
io.save(ifile="/Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/AlbumOfTheYear/spotifyAlbumIDs.p", idata=Series(retval))

In [ ]:
""" Discogs Music DB ID """

__all__ = ["MusicDBID"]

from mdbid import MusicDBIDBase

###########################################################################################################################################
## Genius
###########################################################################################################################################
class MusicDBID(MusicDBIDBase):
    def __init__(self, debug=False):
        super().__init__(debug)
        patterns  = [r'/album/([\w+])']
        patterns  = [r'/artist/([\w+])']
        patterns  = [r'([\w+])']
        self.patterns = patterns
        self.get = self.getArtistID

    def getArtistID(self, s):
        self.s = str(s)
        return self.getArtistIDFromPatterns(self.s, self.patterns)

In [ ]:
m = MusicDBID()

In [ ]:
m.get("http://open.spotify.com/album/5d2JEbUZV4hd5GiQrfbSa5")